# 03. AR·순수 확산·Block diffusion 성능 모델

목표: output 길이, block size, denoising step이 sequential forward-pass 수와 pass당 확정 token에 미치는 영향을 비교합니다. 실제 GPU latency가 아니라 구조적 계산량을 보는 toy model입니다.

In [ ]:
import math

def decoding_stats(output_tokens, mode, block_size=None, denoise_steps=None):
    if mode == "ar":
        passes = output_tokens
    elif mode == "full_diffusion":
        passes = denoise_steps
    elif mode == "block_diffusion":
        blocks = math.ceil(output_tokens / block_size)
        passes = blocks * denoise_steps
    else:
        raise ValueError(mode)
    return {
        "mode": mode,
        "passes": passes,
        "committed_tokens_per_pass": output_tokens / passes,
    }

configs = [
    decoding_stats(1_024, "ar"),
    decoding_stats(1_024, "full_diffusion", denoise_steps=64),
    decoding_stats(1_024, "block_diffusion", block_size=256, denoise_steps=32),
]
for item in configs:
    print(item)

## Block size와 step trade-off

큰 block은 block 수를 줄이지만 한 번에 복원할 불확실성이 커질 수 있습니다. 아래 표는 품질 변화와 step당 FLOPs를 포함하지 않습니다.

In [ ]:
for block_size in (16, 32, 64, 128, 256):
    stats = decoding_stats(1_024, "block_diffusion", block_size=block_size, denoise_steps=24)
    print(f"block={block_size:3d}, passes={stats['passes']:4d}, token/pass={stats['committed_tokens_per_pass']:.2f}")

## Adaptive stopping

각 block이 최대 48 step이 아니라 confidence gap에 따라 일찍 끝난다고 가정합니다.

In [ ]:
def adaptive_block_passes(step_counts, max_steps):
    clipped = [min(step, max_steps) for step in step_counts]
    return sum(clipped), clipped

fixed_passes = 4 * 48
adaptive_passes, used = adaptive_block_passes([18, 27, 32, 21], max_steps=48)
print("fixed passes:", fixed_passes)
print("adaptive steps by block:", used)
print("adaptive passes:", adaptive_passes)
print(f"pass reduction={(1-adaptive_passes/fixed_passes):.1%}")

## Wall-clock 추정에는 pass cost가 필요하다

AR pass는 KV cache로 새 token 중심 계산을 하지만 diffusion pass는 현재 canvas 전체를 다시 계산할 수 있습니다. 가상 pass 비용을 곱해 구조적 pass 수가 그대로 speedup이 아님을 확인합니다.

In [ ]:
def estimated_time_ms(passes, cost_per_pass_ms):
    return passes * cost_per_pass_ms

ar_time = estimated_time_ms(1_024, 0.20)
diffusion_time = estimated_time_ms(adaptive_passes, 2.50)
print(f"AR toy time={ar_time:.1f} ms")
print(f"block diffusion toy time={diffusion_time:.1f} ms")
print("This is illustrative, not a benchmark.")

## 실제 benchmark 설계

동일 hardware·precision·batch·prompt/output 길이에서 time-to-first-token, inter-token latency, end-to-end latency, 확정 token/s, peak memory와 quality를 함께 측정하세요. Candidate prediction이나 재마스킹된 token을 throughput에 포함하지 않습니다.